# 01 · Exploratory Data Analysis (Google Colab version)

**Group 11 · Track 3 Project Showcase · Topic A: Customer Churn Prediction with Simple Uplift Extension**

This notebook runs on its own in Google Colab: **Runtime → Run all**. The setup cells below install the libraries,
recreate the project's `ml/` code, download the public IBM Telco dataset and run the pipeline
(about 1 minute on a standard Colab runtime). The notebook itself starts at the next big heading.

The same notebook lives in the project as `notebooks/01_eda.ipynb`; this copy only adds setup, so every number matches the project.

## Setup 1 · Install the exact library versions the project uses

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'scikit-learn==1.6.1', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-uplift==0.5.1'], check=True)
    print("Installed:", 'scikit-learn==1.6.1', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-uplift==0.5.1')
else:
    print("Not on Colab: using the libraries already installed.")

## Setup 2 · Recreate the project code

These files are copied unchanged from the project's `ml/` folder (and `params.yaml`, the business assumptions).

In [ ]:
from pathlib import Path
for d in ['ml/models', 'ml/results/figures', 'data/raw', 'data/processed', 'backend/data', 'notebooks/figures']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Project folders ready in', Path.cwd())

In [ ]:
%%writefile ml/download_data.py
"""
Download the IBM Telco Customer Churn dataset into data/raw/.

Why a script instead of committing the CSV?
  - Raw data stays out of git (project rule).
  - Anyone can rebuild the data folder with one command.

Run from the project root:
    python ml/download_data.py
"""
from pathlib import Path
import urllib.request

# Public copy of the dataset published by IBM on GitHub.
URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

# Build paths relative to this file so the script works from any folder.
PROJECT_ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "telco_churn.csv"


def main() -> None:
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    if RAW_PATH.exists():
        print(f"Already downloaded: {RAW_PATH}")
        return
    print(f"Downloading from {URL} ...")
    urllib.request.urlretrieve(URL, RAW_PATH)
    print(f"Saved to {RAW_PATH} ({RAW_PATH.stat().st_size / 1024:.0f} KB)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/preprocess.py
"""
Data cleaning + preprocessing for the Telco churn dataset.

Two separate steps:
  1. clean_data()          -> fixes the raw data (types, missing values, target).
                              Output is still human-readable (e.g. "Month-to-month").
  2. build_preprocessor()  -> scikit-learn transformer that ENCODES categories
                              (one-hot) and SCALES numbers (standardisation).

Why keep step 2 separate?
  The scaler must learn its mean/std from the TRAINING data only. If we scaled
  the whole dataset before splitting, information from the test set would
  "leak" into training. Putting the scaler inside a scikit-learn Pipeline
  (Phase 3) avoids this and lets the backend reuse the exact same steps.

Run from the project root to create data/processed/telco_clean.csv:
    python ml/preprocess.py
"""
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "telco_churn.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "telco_clean.csv"

ID_COL = "customerID"
TARGET = "Churn"

# Numeric columns -> get scaled.
NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]

# Already 0/1 -> used as-is (no encoding or scaling needed).
BINARY_NUMERIC_COLS = ["SeniorCitizen"]

# Text categories -> get one-hot encoded.
CATEGORICAL_COLS = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]

FEATURE_COLS = NUMERIC_COLS + BINARY_NUMERIC_COLS + CATEGORICAL_COLS


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of the raw Telco data."""
    df = df.copy()

    # 1) TotalCharges is stored as text and has 11 blank values.
    #    All 11 blanks are brand-new customers (tenure = 0) who have not been
    #    billed yet, so the correct value is 0 (not "missing").
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)

    # 2) "No internet service" / "No phone service" mean the same as "No" for
    #    the add-on columns. The InternetService / PhoneService columns already
    #    record whether the customer has the base service, so merging these
    #    removes duplicate information and gives fewer one-hot columns.
    df = df.replace({"No internet service": "No", "No phone service": "No"})

    # 3) Target: "Yes"/"No" -> 1/0 so models can use it.
    if TARGET in df.columns:
        df[TARGET] = (df[TARGET] == "Yes").astype(int)

    return df


def build_preprocessor() -> ColumnTransformer:
    """
    Encoding + scaling step, to be placed at the start of a model Pipeline.
      - numbers      -> StandardScaler (mean 0, std 1)
      - categories   -> OneHotEncoder (one 0/1 column per category)
      - 0/1 columns  -> passed through unchanged
    handle_unknown="ignore" means an unseen category at prediction time
    (e.g. a typo from the web form) becomes all zeros instead of crashing.
    """
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CATEGORICAL_COLS),
            ("bin", "passthrough", BINARY_NUMERIC_COLS),
        ]
    )


def load_clean() -> pd.DataFrame:
    """Load the cleaned CSV (create it first with `python ml/preprocess.py`)."""
    return pd.read_csv(CLEAN_PATH)


RANDOM_STATE = 42   # fixed seed -> same split / same results every run
TEST_SIZE = 0.20    # 80% train, 20% test


def split_data(df: pd.DataFrame):
    """
    80/20 train/test split, STRATIFIED on Churn so both parts keep the
    same churn rate (about 26.5%). Every phase uses this function, so all
    models are evaluated on exactly the same test customers.
    """
    return train_test_split(
        df, test_size=TEST_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE
    )


def main() -> None:
    raw = pd.read_csv(RAW_PATH)
    clean = clean_data(raw)
    CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    clean.to_csv(CLEAN_PATH, index=False)
    print(f"Raw shape:   {raw.shape}")
    print(f"Clean shape: {clean.shape}")
    print(f"Missing values after cleaning: {int(clean.isna().sum().sum())}")
    print(f"Churn rate: {clean[TARGET].mean():.1%}")
    print(f"Saved to {CLEAN_PATH}")


if __name__ == "__main__":
    main()


## Setup 3 · Download the data and run the pipeline

- `ml/download_data.py`
- `ml/preprocess.py`

Outputs are hidden unless a step fails.

In [ ]:
import subprocess, sys, time
for script in ['ml/download_data.py', 'ml/preprocess.py']:
    t = time.time()
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"{script} failed")
    print(f"✓ {script}  ({time.time() - t:.0f} s)")
print("Setup complete.")

# 01 · Exploratory Data Analysis (EDA): IBM Telco Customer Churn

**Goal:** understand the data before modelling. Which customers leave (churn), and what is different about them?

**Steps**
1. Load and clean the data (missing values, data types)
2. Overall churn rate (is the data imbalanced?)
3. Churn by contract type, tenure and charges
4. Other strong signals (internet type, payment method, add-on services)
5. Correlations
6. Encoding and scaling (how the data is prepared for models)
7. Key takeaways

> Before running: `python ml/download_data.py` and `python ml/preprocess.py` from the project root.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Let the notebook import our own cleaning code from the ml/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "ml"))
from preprocess import (RAW_PATH, NUMERIC_COLS, CATEGORICAL_COLS, FEATURE_COLS,
                        clean_data, build_preprocessor)

FIG_DIR = PROJECT_ROOT / "notebooks" / "figures"
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
CHURN_COLORS = {"Stayed": "#4C72B0", "Churned": "#DD8452"}
pd.set_option("display.max_columns", 30)

## 1. Load the raw data

In [ ]:
raw = pd.read_csv(RAW_PATH)
print("Rows, columns:", raw.shape)
raw.head()

Each row is **one customer**. The columns describe:
- **Who they are:** gender, SeniorCitizen, Partner, Dependents
- **What they buy:** phone/internet services and add-ons (OnlineSecurity, TechSupport, Streaming…)
- **How they pay:** Contract, PaymentMethod, PaperlessBilling, MonthlyCharges, TotalCharges
- **How long they've stayed:** tenure (months)
- **Target:** `Churn` = did the customer leave in the last month? (Yes/No)

### 1a. Data quality check

In [ ]:
print("Data types:")
print(raw.dtypes.value_counts(), "\n")
print("Missing (NaN) values:", int(raw.isna().sum().sum()))
print("Duplicate customer IDs:", int(raw["customerID"].duplicated().sum()))

# TotalCharges should be a number but is stored as text. Find the blanks:
blank = raw["TotalCharges"].str.strip() == ""
print("Blank TotalCharges:", int(blank.sum()))
raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

**What we found:** there are no NaN values, but `TotalCharges` is stored as **text**, and **11 rows are blank**.
Every one of these customers has **tenure = 0**, so they joined this month and haven't been billed yet.

**Fix:** convert `TotalCharges` to a number and fill the blanks with **0**. This is the true value, so we don't need to drop or guess anything.
We also merge "No internet service" / "No phone service" into "No" for the add-on columns, because the `InternetService` / `PhoneService`
columns already record that. The cleaning code lives in `ml/preprocess.py` so the models and the backend use **exactly the same steps**.

In [ ]:
df = clean_data(raw)
print("Missing values after cleaning:", int(df.isna().sum().sum()))
print("TotalCharges type:", df["TotalCharges"].dtype)
print("Churn values:", sorted(df["Churn"].unique()), "(1 = churned)")

# Readable label for charts
df["Status"] = df["Churn"].map({0: "Stayed", 1: "Churned"})

## 2. Overall churn rate

In [ ]:
counts = df["Status"].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values, color=[CHURN_COLORS[s] for s in counts.index])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 50, f"{v:,}\n({v / len(df):.1%})", ha="center")
ax.set_title("How many customers churned?")
ax.set_ylabel("Customers")
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.savefig(FIG_DIR / "01_churn_rate.png", dpi=120); plt.show()

📝 **Note:** about **26.5%** of customers churned (1,869 of 7,043), so roughly **1 in 4**.

The data is **imbalanced**: there are about 3 "stayed" customers for every "churned" one. This matters for modelling:
a lazy model that predicts "nobody churns" would already be **73.5% accurate** while being useless.
That is why in Phase 3 we judge models by **precision, recall, F1 and ROC-AUC** instead of plain accuracy.

## 3a. Churn by contract type

In [ ]:
def churn_rate_plot(col, title, fname, order=None, rotate=0):
    """Bar chart of churn rate (%) per category, with customer counts."""
    stats = df.groupby(col)["Churn"].agg(rate="mean", n="size")
    if order is not None:
        stats = stats.loc[order]
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(stats.index.astype(str), stats["rate"] * 100, color="#DD8452")
    for b, (r, n) in zip(bars, stats.values):
        ax.text(b.get_x() + b.get_width() / 2, r * 100 + 1, f"{r:.0%}\n(n={int(n):,})", ha="center", fontsize=9)
    ax.axhline(df["Churn"].mean() * 100, ls="--", color="grey", label="Overall churn rate")
    ax.set_ylabel("Churn rate (%)"); ax.set_title(title)
    ax.set_ylim(0, stats["rate"].max() * 100 * 1.3)
    ax.legend(); plt.xticks(rotation=rotate)
    plt.tight_layout(); plt.savefig(FIG_DIR / fname, dpi=120); plt.show()
    return stats

churn_rate_plot("Contract", "Churn rate by contract type", "02_contract.png")

📝 **Note:** contract type is the **strongest single signal** in the data.
- **Month-to-month:** about **43%** churn
- **One year:** about **11%**
- **Two year:** about **3%**

A month-to-month customer can leave at any time with no penalty, so they leave far more often. More than half of all customers (3,875) are on month-to-month,
so this group is where most churn happens. One business idea follows directly: **move customers to longer contracts**.

## 3b. Churn by tenure (how long the customer has stayed)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: distribution of tenure for stayers vs churners
sns.histplot(data=df, x="tenure", hue="Status", bins=36, multiple="layer",
             palette=CHURN_COLORS, ax=axes[0])
axes[0].set_title("Tenure distribution: stayed vs churned")
axes[0].set_xlabel("Tenure (months)")

# Right: churn rate per tenure group
df["TenureGroup"] = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 48, 72],
                           labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
rate = df.groupby("TenureGroup", observed=True)["Churn"].mean() * 100
axes[1].bar(rate.index.astype(str), rate.values, color="#DD8452")
for i, v in enumerate(rate.values):
    axes[1].text(i, v + 1, f"{v:.0f}%", ha="center")
axes[1].set_title("Churn rate by tenure group")
axes[1].set_xlabel("Tenure (months)"); axes[1].set_ylabel("Churn rate (%)")
plt.tight_layout(); plt.savefig(FIG_DIR / "03_tenure.png", dpi=120); plt.show()

📝 **Note:** **new customers are the riskiest.**
- In the first **6 months**, about **53%** of customers churn.
- After **4+ years** (49–72 months), churn drops to about **10%**.

The left chart shows churners piled up at low tenure (median **10 months** vs **38 months** for customers who stayed).
In plain words: if a customer gets through the first year, they usually stay. Retention effort should focus on the **early months**.

## 3c. Churn by charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["MonthlyCharges", "TotalCharges"]):
    sns.kdeplot(data=df, x=col, hue="Status", fill=True, common_norm=False,
                palette=CHURN_COLORS, ax=ax)
    ax.set_title(f"{col}: stayed vs churned")
plt.tight_layout(); plt.savefig(FIG_DIR / "04_charges.png", dpi=120); plt.show()

df.groupby("Status")[["MonthlyCharges", "TotalCharges"]].median().round(1)

📝 **Note:**
- **Monthly charges:** churners pay **more per month** (median about **$80** vs **$64**). The churn curve peaks at roughly $70–$105,
  which matches fiber-optic internet plans (see section 4). High bills combined with no long-term contract is a risky mix.
- **Total charges:** churners have **lower** totals (median about **$704** vs **$1,680**). This is not because cheap customers leave.
  Total charges ≈ monthly charges × tenure, and churners simply haven't been around long. So TotalCharges is mostly a **tenure effect**.

## 4. Other strong signals

In [ ]:
churn_rate_plot("InternetService", "Churn rate by internet service", "05_internet.png")
churn_rate_plot("PaymentMethod", "Churn rate by payment method", "06_payment.png", rotate=15)

📝 **Note:**
- **Fiber optic** customers churn at about **42%**, more than double DSL (**19%**). Fiber is the expensive plan,
  so this could point to price or service-quality complaints. That's worth investigating as a business question.
- **Electronic check** payers churn at about **45%**, roughly three times the rate of automatic bank/card payments (**15–17%**).
  Automatic payment is "set and forget". Customers who pay by check make an active choice every month, which gives them a regular moment to reconsider.

In [ ]:
# Add-on services and customer profile: churn rate when the customer has the feature vs doesn't
cols = ["OnlineSecurity", "TechSupport", "OnlineBackup", "DeviceProtection",
        "Partner", "Dependents", "PaperlessBilling"]
rows = []
for c in cols:
    r = df.groupby(c)["Churn"].mean()
    rows.append({"feature": c, "Yes": r.get("Yes"), "No": r.get("No")})
rows.append({"feature": "SeniorCitizen", "Yes": df.loc[df.SeniorCitizen == 1, "Churn"].mean(),
             "No": df.loc[df.SeniorCitizen == 0, "Churn"].mean()})
feat = pd.DataFrame(rows).set_index("feature") * 100

ax = feat.plot.barh(figsize=(8, 5), color={"Yes": "#55A868", "No": "#C44E52"})
ax.set_xlabel("Churn rate (%)"); ax.set_title("Churn rate: has feature (Yes) vs not (No)")
ax.axvline(df["Churn"].mean() * 100, ls="--", color="grey")
plt.tight_layout(); plt.savefig(FIG_DIR / "07_addons.png", dpi=120); plt.show()
feat.round(1)

📝 **Note:**
- Customers **with** OnlineSecurity or TechSupport churn about **half as often** (about 15% vs 31%). Support services seem to make customers "sticky".
  (This is correlation, not proof of cause. These customers may differ in other ways too.)
- Customers with a **partner or dependents** churn less. Family accounts are harder to move.
- **Senior citizens** (about 42%) and **paperless billing** users (about 34%) churn more than average.
- **Gender** (not shown) makes almost no difference: about 27% vs 26%.

## 5. Correlations

In [ ]:
# One-hot encode categories so every feature becomes numeric, then
# measure how strongly each one moves with Churn (Pearson correlation).
encoded = pd.get_dummies(df[FEATURE_COLS], drop_first=False, dtype=int)
encoded["Churn"] = df["Churn"]
corr_with_churn = encoded.corr()["Churn"].drop("Churn").sort_values()

top = pd.concat([corr_with_churn.head(8), corr_with_churn.tail(8)])
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color=["#4C72B0" if v < 0 else "#DD8452" for v in top.values])
ax.set_title("Features most correlated with churn\n(orange = more churn, blue = less churn)")
ax.set_xlabel("Correlation with Churn")
plt.tight_layout(); plt.savefig(FIG_DIR / "08_corr_churn.png", dpi=120); plt.show()

📝 **Note:** this chart ranks every feature by how strongly it moves with churn (from −1 to +1).
- **Raise churn (orange):** month-to-month contract, no online security, no tech support, fiber optic, electronic check.
- **Lower churn (blue):** two-year contract, long tenure, no internet service, one-year contract.

These match the earlier charts, which is a good sanity check. Correlations of about 0.3–0.4 are only **moderate**, though.
No single feature predicts churn on its own, which is why we combine them in a model.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df[NUMERIC_COLS + ["SeniorCitizen", "Churn"]].corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation between numeric columns")
plt.tight_layout(); plt.savefig(FIG_DIR / "09_corr_numeric.png", dpi=120); plt.show()

📝 **Note:** **TotalCharges is strongly correlated with tenure (0.83)** and with MonthlyCharges (0.65). That makes sense, since total ≈ monthly × months.
Features that overlap this much carry partly the same information (*multicollinearity*). It can make logistic-regression coefficients harder to interpret,
but it doesn't hurt the predictions much, and tree models (Random Forest, XGBoost) handle it fine. We keep all three and note this for the viva.

## 6. Encoding and scaling

Models need **numbers**, and some need numbers on a **similar scale**. `build_preprocessor()` in `ml/preprocess.py` does:

| Column type | Example | Transformation |
|---|---|---|
| Numeric | tenure, MonthlyCharges, TotalCharges | **StandardScaler**: (x − mean) / std → mean 0, std 1 |
| Category | Contract, PaymentMethod… | **One-hot encoding**: one 0/1 column per category |
| Already 0/1 | SeniorCitizen | passed through unchanged |

**Why scale?** Without it, TotalCharges (up to about 8,700) would dwarf tenure (up to 72) for logistic regression. Tree models don't need it, but it doesn't hurt them.

**Important:** in Phase 3 this step goes **inside a Pipeline** and is fitted on the **training set only**. Fitting it on all the data
would let test-set information leak into training.

In [ ]:
pre = build_preprocessor()
X = pre.fit_transform(df[FEATURE_COLS])   # demo only: fitted on ALL data here just to show the output
X = pd.DataFrame(X, columns=pre.get_feature_names_out())
print("Before:", df[FEATURE_COLS].shape, "->  After:", X.shape)
X[["num__tenure", "num__MonthlyCharges", "num__TotalCharges"]].describe().loc[["mean", "std", "min", "max"]].round(2)

📝 **Note:** 19 original columns become **26 numeric columns**. After scaling, the numeric columns have **mean ≈ 0 and std ≈ 1**, so they are now on the same scale.

## 7. Key takeaways

1. **Churn rate is 26.5%**, so the data is imbalanced. Use precision/recall/F1/ROC-AUC, not accuracy.
2. **Contract type** is the biggest driver: month-to-month 43% vs two-year 3%.
3. **New customers** (first 6 months) churn the most, at about 53%.
4. **Higher monthly bills, fiber optic and electronic check** are linked to higher churn.
5. **Security/support add-ons, partners and dependents** are linked to lower churn.
6. TotalCharges largely repeats tenure (correlation 0.83).
7. Data is clean: 11 blank TotalCharges values were filled with 0 (new customers), and nothing was dropped.

**Next (Phase 3):** train Logistic Regression, Random Forest and XGBoost on these features and see whether the models agree with these EDA findings (using SHAP).